# RT-DETR-L: Baseline 5-Fold Cross-Validation

## Overview

This notebook documents our initial RT-DETR-L experiment using pretrained weights and five-fold video-level cross-validation.

## Methodology

- **Architecture:** RT-DETR-L
- **Pretrained checkpoint:** `rtdetr-l.pt`
- **Dataset:** VisDrone, 8 object classes
- **Validation:** 5 folds, grouped by video sequence
- **Training:** 4 epochs per fold
- **Image size:** 960px
- **Batch size:** 1
- **Optimizer:** AdamW
- **Learning rate:** 0.001
- **Random seed:** 42
- **Evaluation:** COCOeval with `maxDets=[1,100,500]`

Training and validation frames were separated at the video level to reduce data leakage.

## Results

| Fold | Validation images | mAP@100 |
|---|---:|---:|
| 0 | 1,142 | 0.06540 |
| 1 | 992 | 0.05236 |
| 2 | 1,285 | 0.05789 |
| 3 | 1,080 | 0.04185 |
| 4 | 940 | 0.07533 |

**Global OOF results:**

| Metric | Value |
|---|---:|
| mAP@100 | **0.050827** |
| AP50@500 | 0.104572 |
| AP_small@500 | 0.020810 |

The baseline experiment motivated further RT-DETR-L tuning, which was documented in the separate `RT-DETR-L Tuned 5-Fold CV` notebook.

## Reproducibility

The experiment was performed in Google Colab using `ultralytics==8.4.202`, PyTorch and `pycocotools`.

The prepared dataset and generated checkpoints were stored on Google Drive. These files are not included in the GitHub repository.

Reproducing the experiment requires updating the Google Drive paths and supplying the same prepared dataset.

In [ ]:
import os
import json
import random
import zipfile
import torch
import numpy as np

from pathlib import Path

# ==========================================
# 1. PROJECT PATHS
# ==========================================

ROOT = Path("/content/drive_rtdetr_new/MyDrive")

ZIP_PATH = ROOT / "hw2/data_prepared.zip"

RESULTS_DIR = ROOT / "hw2_rtdetr"
CONFIG_DIR = RESULTS_DIR / "configs"
CV_COCO_DIR = RESULTS_DIR / "cv_annotations"

PREP_DIR = Path("/content/hw2/data_prepared")
WORK_DIR = Path("/content/hw2/rtdetr_cv")

# Create directories on the new account
for directory in [
    RESULTS_DIR,
    CONFIG_DIR,
    CV_COCO_DIR,
    WORK_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. REPRODUCIBILITY
# ==========================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ==========================================
# 3. EXPERIMENT PARAMETERS
# ==========================================

CLASSES = [
    "person",
    "bicycle",
    "car",
    "van",
    "truck",
    "tricycle",
    "bus",
    "motor"
]

NUM_CLASSES = 8
NUM_FOLDS = 5

IMG_SIZE_CV = 960
IMG_SIZE_FULL = 1280

EPOCHS_CV = 4
EPOCHS_FULL = 8

MODEL_NAME = "rtdetr-l.pt"

# ==========================================
# 4. CHECK ENVIRONMENT
# ==========================================

assert ZIP_PATH.exists(), "Dataset ZIP not found!"
assert torch.cuda.is_available(), "GPU not available!"

with zipfile.ZipFile(ZIP_PATH) as z:
    assert z.testzip() is None, "Corrupted ZIP archive!"

print("RT-DETR ENVIRONMENT READY")
print("=" * 50)

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:",
      round(
          torch.cuda.get_device_properties(0).total_memory
          / 1024**3, 2
      ),
      "GB")

print("Dataset:", ZIP_PATH)
print("Results:", RESULTS_DIR)
print("Model:", MODEL_NAME)
print("CV folds:", NUM_FOLDS)

print("\nREADY FOR DATASET PREPARATION!")

RT-DETR ENVIRONMENT READY
GPU: Tesla T4
GPU memory: 14.56 GB
Dataset: /content/drive_rtdetr_new/MyDrive/hw2/data_prepared.zip
Results: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr
Model: rtdetr-l.pt
CV folds: 5

READY FOR DATASET PREPARATION!


In [ ]:
import zipfile
import shutil
import time
from pathlib import Path

start = time.perf_counter()

# Paths from Cell 1
EXTRACT_ROOT = Path("/content/hw2")
PREP_DIR = EXTRACT_ROOT / "data_prepared"

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

print("EXTRACTING VISDRONE DATASET")
print("=" * 55)

extracted = 0
skipped = 0

with zipfile.ZipFile(ZIP_PATH, "r") as z:

    files = []

    for info in z.infolist():

        if info.is_dir():
            continue

        parts = Path(info.filename).parts

        if "data_prepared" not in parts:
            continue

        index = parts.index("data_prepared")
        relative = Path(*parts[index:])

        # Protect extraction paths
        if ".." in relative.parts:
            raise ValueError(f"Unsafe path: {info.filename}")

        files.append((info, relative))

    assert len(files) > 10000, (
        f"Unexpected archive structure: {len(files)} files"
    )

    print("Files to check:", len(files))

    for info, relative in files:

        target = EXTRACT_ROOT / relative

        # Skip already complete files
        if (
            target.exists()
            and target.stat().st_size == info.file_size
        ):
            skipped += 1
            continue

        target.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with z.open(info) as src:
            with open(target, "wb") as dst:
                shutil.copyfileobj(
                    src,
                    dst,
                    length=1024 * 1024
                )

        extracted += 1

        if extracted % 1000 == 0:
            print(f"Extracted: {extracted} files")

# Verify dataset
images = list(
    (PREP_DIR / "images").rglob("*.jpg")
)

labels = list(
    (PREP_DIR / "labels").rglob("*.txt")
)

coco_files = list(
    (PREP_DIR / "coco").glob("fold*_*.json")
)

print("\nDATASET EXTRACTION COMPLETED")
print("=" * 55)

print("Images:", len(images))
print("Labels:", len(labels))
print("COCO annotation files:", len(coco_files))

print("New files extracted:", extracted)
print("Existing files skipped:", skipped)

print("Dataset location:", PREP_DIR)

print(
    "Time:",
    round(time.perf_counter() - start, 1),
    "seconds"
)

assert len(images) == 5439, "Incorrect number of images!"
assert len(labels) == 5439, "Incorrect number of labels!"
assert len(coco_files) == 10, "Missing COCO annotations!"

print("\nVISDRONE DATASET READY!")

EXTRACTING VISDRONE DATASET
Files to check: 10905
Extracted: 1000 files
Extracted: 2000 files
Extracted: 3000 files
Extracted: 4000 files
Extracted: 5000 files
Extracted: 6000 files
Extracted: 7000 files
Extracted: 8000 files
Extracted: 9000 files
Extracted: 10000 files

DATASET EXTRACTION COMPLETED
Images: 5439
Labels: 5439
COCO annotation files: 10
New files extracted: 10905
Existing files skipped: 0
Dataset location: /content/hw2/data_prepared
Time: 36.5 seconds

VISDRONE DATASET READY!


In [ ]:
import json
import shutil
import yaml
import pandas as pd

from pathlib import Path

# Paths from Cell 1
PREP_DIR = Path("/content/hw2/data_prepared")
RESULTS_DIR = Path(
    "/content/drive_rtdetr_new/MyDrive/hw2_rtdetr"
)

CONFIG_DIR = RESULTS_DIR / "configs"
CV_COCO_DIR = RESULTS_DIR / "cv_annotations"

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CV_COCO_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = [
    "person", "bicycle", "car", "van",
    "truck", "tricycle", "bus", "motor"
]

EXPECTED_TRAIN = [2161, 2235, 2088, 2191, 2261]
EXPECTED_VAL = [1142, 992, 1285, 1080, 940]

def frame_number(filename):
    return int(Path(filename).stem.rsplit("_", 1)[-1])

def video_name(filename):
    return Path(filename).stem.rsplit("_", 1)[0]

def save_yaml(train_paths, val_paths, name):
    train_txt = CONFIG_DIR / f"{name}_train.txt"
    val_txt = CONFIG_DIR / f"{name}_val.txt"

    train_txt.write_text("\n".join(train_paths) + "\n")
    val_txt.write_text("\n".join(val_paths) + "\n")

    yaml_path = CONFIG_DIR / f"{name}.yaml"

    with open(yaml_path, "w") as f:
        yaml.safe_dump({
            "path": str(PREP_DIR),
            "train": str(train_txt),
            "val": str(val_txt),
            "nc": 8,
            "names": CLASSES
        }, f, sort_keys=False)

    return yaml_path

summary = []

for fold in range(5):

    # Load original COCO annotations
    with open(PREP_DIR / f"coco/fold{fold}_train.json") as f:
        train_coco = json.load(f)

    with open(PREP_DIR / f"coco/fold{fold}_val.json") as f:
        val_coco = json.load(f)

    # Same frame sampling as YOLO CV
    selected_train = [
        img for img in train_coco["images"]
        if (frame_number(img["file_name"]) - 1) % 10 == 0
    ]

    val_images = val_coco["images"]

    assert len(selected_train) == EXPECTED_TRAIN[fold]
    assert len(val_images) == EXPECTED_VAL[fold]

    # Verify video separation
    train_videos = {
        video_name(img["file_name"])
        for img in train_coco["images"]
    }

    val_videos = {
        video_name(img["file_name"])
        for img in val_images
    }

    assert train_videos.isdisjoint(val_videos)

    # Build image paths
    train_paths = [
        str(PREP_DIR / img["file_name"])
        for img in selected_train
    ]

    val_paths = [
        str(PREP_DIR / img["file_name"])
        for img in val_images
    ]

    assert all(Path(p).is_file() for p in train_paths + val_paths)

    # Save CV YAML
    yaml_path = save_yaml(
        train_paths,
        val_paths,
        f"rtdetr_fold{fold}_cv"
    )

    # Filter COCO annotations for selected training images
    selected_ids = {img["id"] for img in selected_train}

    train_cv_coco = dict(train_coco)
    train_cv_coco["images"] = selected_train
    train_cv_coco["annotations"] = [
        ann for ann in train_coco["annotations"]
        if ann["image_id"] in selected_ids
    ]

    with open(
        CV_COCO_DIR / f"fold{fold}_train_cv10.json", "w"
    ) as f:
        json.dump(train_cv_coco, f)

    # Preserve original validation annotations
    shutil.copy2(
        PREP_DIR / f"coco/fold{fold}_val.json",
        CV_COCO_DIR / f"fold{fold}_val.json"
    )

    summary.append({
        "fold": fold,
        "train_images": len(selected_train),
        "val_images": len(val_images),
        "video_overlap": 0,
        "yaml_exists": yaml_path.exists()
    })

    # Full training configuration for Fold 0
    if fold == 0:
        full_train_paths = [
            str(PREP_DIR / img["file_name"])
            for img in train_coco["images"]
        ]

        full_yaml = save_yaml(
            full_train_paths,
            val_paths,
            "rtdetr_fold0_full"
        )

# Save summary
summary_df = pd.DataFrame(summary)
summary_df.to_csv(
    RESULTS_DIR / "rtdetr_cv_splits_summary.csv",
    index=False
)

print("\nRT-DETR CONFIGURATIONS RESTORED")
print("=" * 55)

display(summary_df)

print("CV YAML files:", 5)
print("Full training YAML:", full_yaml)
print("COCO annotations:", CV_COCO_DIR)
print("GPU available:", torch.cuda.is_available())

print("\nREADY FOR MODEL INSTALLATION!")


RT-DETR CONFIGURATIONS RESTORED


,fold,train_images,val_images,video_overlap,yaml_exists
0,0,2161,1142,0,True
1,1,2235,992,0,True
2,2,2088,1285,0,True
3,3,2191,1080,0,True
4,4,2261,940,0,True


CV YAML files: 5
Full training YAML: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/configs/rtdetr_fold0_full.yaml
COCO annotations: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/cv_annotations
GPU available: True

READY FOR MODEL INSTALLATION!


In [ ]:
%pip install -q ultralytics==8.4.202

In [ ]:
import shutil
import torch
import ultralytics

from pathlib import Path
from ultralytics import RTDETR

# Save pretrained checkpoint permanently on Google Drive
WEIGHTS_PATH = RESULTS_DIR / "rtdetr-l.pt"

print("RT-DETR MODEL SETUP")
print("=" * 55)

print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "GPU is required!"

# Reuse checkpoint if previously downloaded
if WEIGHTS_PATH.exists():
    print("Loading saved pretrained weights...")
    model = RTDETR(str(WEIGHTS_PATH))

else:
    print("Downloading pretrained RT-DETR-L...")
    model = RTDETR("rtdetr-l.pt")

    downloaded = Path("rtdetr-l.pt")
    assert downloaded.exists(), "Downloaded weights not found"

    shutil.copy2(downloaded, WEIGHTS_PATH)

print("\nMODEL LOADED SUCCESSFULLY")
print("=" * 55)

print("Model:", model.model.__class__.__name__)
print("Pretrained classes:", len(model.names))
print("Weights:", WEIGHTS_PATH)
print(
    "Checkpoint size:",
    round(WEIGHTS_PATH.stat().st_size / 1024**2, 2),
    "MB"
)

print("GPU:", torch.cuda.get_device_name(0))
print("\nREADY FOR GPU MEMORY TEST!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
RT-DETR MODEL SETUP
Ultralytics: 8.4.202
CUDA available: True

MODEL LOADED SUCCESSFULLY
Model: DetectionModel
Pretrained classes: 80
Weights: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/rtdetr-l.pt
Checkpoint size: 63.43 MB
GPU: Tesla T4

READY FOR GPU MEMORY TEST!


In [ ]:
import gc
import time
import yaml
import torch

from pathlib import Path
from ultralytics import RTDETR

# ==========================================
# 1. CREATE A SMALL TEST DATASET
# ==========================================

WEIGHTS_PATH = RESULTS_DIR / "rtdetr-l.pt"
FOLD_YAML = CONFIG_DIR / "rtdetr_fold0_cv.yaml"

TEST_DIR = Path("/content/rtdetr_memory_test")
TEST_DIR.mkdir(parents=True, exist_ok=True)

with open(FOLD_YAML) as f:
    fold_config = yaml.safe_load(f)

train_images = [
    line.strip()
    for line in Path(fold_config["train"]).read_text().splitlines()
    if line.strip()
]

val_images = [
    line.strip()
    for line in Path(fold_config["val"]).read_text().splitlines()
    if line.strip()
]

# Only six training images for the memory test
test_train = train_images[:6]
test_val = val_images[:2]

assert len(test_train) == 6
assert len(test_val) == 2
assert all(Path(p).exists() for p in test_train + test_val)

train_txt = TEST_DIR / "train.txt"
val_txt = TEST_DIR / "val.txt"

train_txt.write_text("\n".join(test_train) + "\n")
val_txt.write_text("\n".join(test_val) + "\n")

test_yaml = TEST_DIR / "dataset.yaml"

with open(test_yaml, "w") as f:
    yaml.safe_dump({
        "path": str(PREP_DIR),
        "train": str(train_txt),
        "val": str(val_txt),
        "nc": 8,
        "names": CLASSES
    }, f, sort_keys=False)

# ==========================================
# 2. LOAD MODEL
# ==========================================

assert torch.cuda.is_available()

model = RTDETR(str(WEIGHTS_PATH))

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(0)

print("RT-DETR-L GPU MEMORY TEST")
print("=" * 55)
print("GPU:", torch.cuda.get_device_name(0))
print("Image size: 960")
print("Batch size: 1")
print("Test images:", len(test_train))
print("Epochs: 1")

start = time.perf_counter()
test_success = False

# ==========================================
# 3. SHORT TRAINING TEST
# ==========================================

try:
    model.train(
        data=str(test_yaml),
        imgsz=960,
        epochs=1,
        batch=1,
        device=0,
        workers=0,
        optimizer="AdamW",
        lr0=0.001,
        seed=42,
        deterministic=False,
        amp=True,
        cache=False,
        val=False,
        save=False,
        plots=False,
        project=str(TEST_DIR),
        name="t4_probe",
        exist_ok=True
    )

    test_success = True

except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("\nCUDA OUT OF MEMORY")
        print("RT-DETR-L does not fit with current settings.")
    else:
        raise

finally:
    elapsed = time.perf_counter() - start

    peak_memory = (
        torch.cuda.max_memory_allocated(0) / 1024**3
    )

    peak_reserved = (
        torch.cuda.max_memory_reserved(0) / 1024**3
    )

    print("\nGPU MEMORY TEST RESULTS")
    print("=" * 55)
    print("Training successful:", test_success)
    print("Peak allocated GPU memory:", round(peak_memory, 2), "GB")
    print("Peak reserved GPU memory:", round(peak_reserved, 2), "GB")
    print("Elapsed time:", round(elapsed, 1), "seconds")

    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\nMEMORY TEST COMPLETED")

RT-DETR-L GPU MEMORY TEST
GPU: Tesla T4
Image size: 960
Batch size: 1
Test images: 6
Epochs: 1
New https://pypi.org/project/ultralytics/8.4.207 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/rtdetr_memory_test/dataset.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_wid

In [ ]:
import torch
import pandas as pd
from pathlib import Path
from ultralytics import RTDETR

# ==========================================
# 1. TRAINING PATHS
# ==========================================

WEIGHTS_PATH = RESULTS_DIR / "rtdetr-l.pt"

DATA_YAML = (
    CONFIG_DIR / "rtdetr_fold0_cv.yaml"
)

PROJECT_DIR = RESULTS_DIR / "cv_runs"
RUN_NAME = "baseline_cv_fold0"

RUN_DIR = PROJECT_DIR / RUN_NAME
LAST_CHECKPOINT = RUN_DIR / "weights/last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

assert WEIGHTS_PATH.exists()
assert DATA_YAML.exists()
assert torch.cuda.is_available()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. RESUME OR START TRAINING
# ==========================================

TARGET_EPOCHS = 4

completed_epochs = 0

if RESULTS_CSV.exists():
    previous_results = pd.read_csv(RESULTS_CSV)
    completed_epochs = len(previous_results)

print("RT-DETR-L BASELINE TRAINING")
print("=" * 55)
print("Fold: 0")
print("Epochs completed:", completed_epochs)
print("Target epochs:", TARGET_EPOCHS)
print("GPU:", torch.cuda.get_device_name(0))
print("Save directory:", RUN_DIR)

if completed_epochs >= TARGET_EPOCHS:
    print("\nTraining already completed. Skipping.")

elif LAST_CHECKPOINT.exists():
    print("\nResuming from saved checkpoint...")

    model = RTDETR(str(LAST_CHECKPOINT))

    model.train(
        resume=True
    )

else:
    print("\nStarting pretrained RT-DETR-L training...")

    model = RTDETR(str(WEIGHTS_PATH))

    model.train(
        data=str(DATA_YAML),
        imgsz=960,
        epochs=4,
        batch=1,
        device=0,
        optimizer="AdamW",
        lr0=0.001,
        seed=42,
        deterministic=False,
        amp=True,
        scale=0.5,
        workers=2,
        cache=False,
        val=False,
        save=True,
        save_period=1,
        plots=False,
        project=str(PROJECT_DIR),
        name=RUN_NAME,
        exist_ok=True
    )

print("\nFOLD 0 TRAINING FINISHED")
print("=" * 55)
print("Checkpoint:", RUN_DIR / "weights/best.pt")
print("Training results:", RESULTS_CSV)

RT-DETR-L BASELINE TRAINING
Fold: 0
Epochs completed: 0
Target epochs: 4
GPU: Tesla T4
Save directory: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/cv_runs/baseline_cv_fold0

Starting pretrained RT-DETR-L training...
New https://pypi.org/project/ultralytics/8.4.207 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive_rtdetr_new/MyDrive/hw2_rtdetr/configs/rtdetr_fold0_cv.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=4, erasing=0.4, exis

In [ ]:
%pip install -q pycocotools

import json
import time
import numpy as np
import torch

from pathlib import Path
from ultralytics import RTDETR
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ==========================================
# 1. PATHS
# ==========================================

RUN_DIR = RESULTS_DIR / "cv_runs/baseline_cv_fold0"

BEST_PT = RUN_DIR / "weights/best.pt"
GT_JSON = PREP_DIR / "coco/fold0_val.json"

CHUNK_DIR = RUN_DIR / "prediction_chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

PRED_JSON = RESULTS_DIR / "baseline_cv_fold0_val_dets.json"
METRICS_JSON = RESULTS_DIR / "baseline_cv_fold0_metrics.json"

assert BEST_PT.exists()
assert GT_JSON.exists()
assert torch.cuda.is_available()

# ==========================================
# 2. LOAD MODEL AND ANNOTATIONS
# ==========================================

model = RTDETR(str(BEST_PT))
coco_gt = COCO(str(GT_JSON))

images = sorted(
    coco_gt.dataset["images"],
    key=lambda img: str(img["id"])
)

print("RT-DETR-L COCO EVALUATION")
print("=" * 55)
print("Checkpoint:", BEST_PT)
print("Validation images:", len(images))
print("Image size: 960")
print("Confidence: 0.01")
print("Max detections: 500")

assert len(images) == 1142

# ==========================================
# 3. GPU INFERENCE IN RESUMABLE CHUNKS
# ==========================================

start = time.perf_counter()
chunk_size = 50
all_detections = []

for chunk_start in range(0, len(images), chunk_size):

    chunk_idx = chunk_start // chunk_size
    chunk_images = images[
        chunk_start:chunk_start + chunk_size
    ]

    chunk_path = CHUNK_DIR / f"chunk_{chunk_idx:03d}.json"

    if chunk_path.exists():
        with open(chunk_path) as f:
            chunk_data = json.load(f)

        assert chunk_data["image_ids"] == [
            img["id"] for img in chunk_images
        ]

        detections = chunk_data["detections"]

    else:
        detections = []

        for img in chunk_images:

            image_path = PREP_DIR / img["file_name"]
            assert image_path.exists(), image_path

            result = model.predict(
                source=str(image_path),
                imgsz=960,
                conf=0.01,
                iou=0.7,
                max_det=500,
                device=0,
                verbose=False
            )[0]

            height, width = result.orig_shape

            assert width == img["width"]
            assert height == img["height"]

            for box in result.boxes:

                cls = int(box.cls.item())
                score = float(box.conf.item())

                x1, y1, x2, y2 = box.xyxy[0].tolist()

                x1 = max(0.0, min(x1, width))
                x2 = max(0.0, min(x2, width))
                y1 = max(0.0, min(y1, height))
                y2 = max(0.0, min(y2, height))

                w = x2 - x1
                h = y2 - y1

                if w <= 0 or h <= 0:
                    continue

                assert 0 <= cls < 8

                detections.append({
                    "image_id": img["id"],
                    "category_id": cls + 1,
                    "bbox": [x1, y1, w, h],
                    "score": score
                })

        # Save completed chunk to Google Drive
        with open(chunk_path, "w") as f:
            json.dump({
                "image_ids": [
                    img["id"] for img in chunk_images
                ],
                "detections": detections
            }, f)

    all_detections.extend(detections)

    print(
        f"Processed {min(chunk_start + chunk_size, len(images))}"
        f"/{len(images)} images"
    )

# Save all COCO detections
with open(PRED_JSON, "w") as f:
    json.dump(all_detections, f)

inference_seconds = time.perf_counter() - start

print("\nINFERENCE COMPLETED")
print("Detections:", len(all_detections))
print("Elapsed time:", round(inference_seconds, 1), "s")
print("Predictions saved:", PRED_JSON)

# ==========================================
# 4. OFFICIAL COCO EVALUATION
# ==========================================

assert len(all_detections) > 0

coco_dt = coco_gt.loadRes(str(PRED_JSON))

evaluator = COCOeval(coco_gt, coco_dt, "bbox")
evaluator.params.imgIds = [
    img["id"] for img in images
]
evaluator.params.maxDets = [1, 100, 500]

evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

# ==========================================
# 5. EXTRACT COMPARABLE AP METRICS
# ==========================================

precision = evaluator.eval["precision"]

def calculate_ap(area_idx=0, maxdet_idx=2, iou_idx=None):

    values = precision

    if iou_idx is not None:
        values = values[iou_idx:iou_idx + 1]

    values = values[:, :, :, area_idx, maxdet_idx]

    valid = values[values > -1]

    return float(valid.mean()) if valid.size else None

metrics = {
    "model": "RT-DETR-L",
    "experiment": "baseline",
    "mode": "cv",
    "fold": 0,
    "epochs": 4,
    "image_size": 960,
    "validation_images": len(images),
    "detections": len(all_detections),
    "mAP_maxDets100": calculate_ap(0, 1),
    "mAP_maxDets500": calculate_ap(0, 2),
    "AP50_maxDets500": calculate_ap(0, 2, 0),
    "AP_small": calculate_ap(1, 2),
    "AP_medium": calculate_ap(2, 2),
    "AP_large": calculate_ap(3, 2)
}

with open(METRICS_JSON, "w") as f:
    json.dump(metrics, f, indent=2)

print("\nRT-DETR-L OFFICIAL COCO RESULTS")
print("=" * 55)

for key, value in metrics.items():
    print(f"{key}: {value}")

print("\nMetrics saved:", METRICS_JSON)
print("\nFOLD 0 COCO EVALUATION COMPLETED!")

loading annotations into memory...
Done (t=0.22s)
creating index...
index created!
RT-DETR-L COCO EVALUATION
Checkpoint: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/cv_runs/baseline_cv_fold0/weights/best.pt
Validation images: 1142
Image size: 960
Confidence: 0.01
Max detections: 500
Processed 50/1142 images
Processed 100/1142 images
Processed 150/1142 images
Processed 200/1142 images
Processed 250/1142 images
Processed 300/1142 images
Processed 350/1142 images
Processed 400/1142 images
Processed 450/1142 images
Processed 500/1142 images
Processed 550/1142 images
Processed 600/1142 images
Processed 650/1142 images
Processed 700/1142 images
Processed 750/1142 images
Processed 800/1142 images
Processed 850/1142 images
Processed 900/1142 images
Processed 950/1142 images
Processed 1000/1142 images
Processed 1050/1142 images
Processed 1100/1142 images
Processed 1142/1142 images

INFERENCE COMPLETED
Detections: 342600
Elapsed time: 137.1 s
Predictions saved: /content/drive_rtdetr_new/MyDrive

In [ ]:
import torch
import pandas as pd

from pathlib import Path
from ultralytics import RTDETR

FOLD = 1
EXPERIMENT = "baseline"

DATA_YAML = (
    CONFIG_DIR / f"rtdetr_fold{FOLD}_cv.yaml"
)

WEIGHTS_PATH = RESULTS_DIR / "rtdetr-l.pt"

PROJECT_DIR = RESULTS_DIR / "cv_runs"
RUN_NAME = f"{EXPERIMENT}_cv_fold{FOLD}"

RUN_DIR = PROJECT_DIR / RUN_NAME

LAST_PT = RUN_DIR / "weights/last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

assert DATA_YAML.exists()
assert WEIGHTS_PATH.exists()
assert torch.cuda.is_available()


completed_epochs = 0

if RESULTS_CSV.exists():
    completed_epochs = len(
        pd.read_csv(RESULTS_CSV)
    )

print("RT-DETR-L BASELINE — FOLD 1")
print("=" * 55)

print("Fold:", FOLD)
print("Experiment:", EXPERIMENT)
print("Completed epochs:", completed_epochs)
print("Target epochs: 4")
print("GPU:", torch.cuda.get_device_name(0))
print("Output:", RUN_DIR)

if completed_epochs >= 4:

    print("\nFold 1 already trained. Skipping.")

elif LAST_PT.exists():

    print("\nResuming Fold 1...")

    model = RTDETR(str(LAST_PT))

    model.train(resume=True)

else:

    print("\nStarting Fold 1 from pretrained weights...")

    model = RTDETR(str(WEIGHTS_PATH))

    model.train(
        data=str(DATA_YAML),
        imgsz=960,
        epochs=4,
        batch=1,
        device=0,
        optimizer="AdamW",
        lr0=0.001,
        seed=42,
        deterministic=False,
        amp=True,
        scale=0.5,
        workers=2,
        cache=False,
        val=False,
        save=True,
        save_period=1,
        plots=False,
        project=str(PROJECT_DIR),
        name=RUN_NAME,
        exist_ok=True
    )

print("\nFOLD 1 TRAINING STATUS")
print("=" * 55)

print("Results CSV:", RESULTS_CSV.exists())
print("Best checkpoint:",
      (RUN_DIR / "weights/best.pt").exists())

if RESULTS_CSV.exists():
    results = pd.read_csv(RESULTS_CSV)
    print("Completed epochs:", len(results))

print("\nFOLD 1 TRAINING STEP COMPLETED")

RT-DETR-L BASELINE — FOLD 1
Fold: 1
Experiment: baseline
Completed epochs: 0
Target epochs: 4
GPU: Tesla T4
Output: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/cv_runs/baseline_cv_fold1

Starting Fold 1 from pretrained weights...
New https://pypi.org/project/ultralytics/8.4.207 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive_rtdetr_new/MyDrive/hw2_rtdetr/configs/rtdetr_fold1_cv.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=4, era

In [ ]:
import json
import os
import time
import numpy as np
import torch

from pathlib import Path
from ultralytics import RTDETR
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ==========================================
# 1. CONFIGURATION
# ==========================================

FOLD = 1
EXPERIMENT = "baseline"

RESULTS_DIR = Path(
    "/content/drive_rtdetr_new/MyDrive/hw2_rtdetr"
)
PREP_DIR = Path("/content/hw2/data_prepared")

RUN_DIR = RESULTS_DIR / "cv_runs" / f"{EXPERIMENT}_cv_fold{FOLD}"
BEST_PT = RUN_DIR / "weights/best.pt"

GT_JSON = PREP_DIR / f"coco/fold{FOLD}_val.json"

CHUNK_DIR = RUN_DIR / "prediction_chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

PRED_JSON = RESULTS_DIR / f"{EXPERIMENT}_cv_fold{FOLD}_val_dets.json"
METRICS_JSON = RESULTS_DIR / f"{EXPERIMENT}_cv_fold{FOLD}_metrics.json"

assert BEST_PT.exists()
assert GT_JSON.exists()
assert torch.cuda.is_available()

# ==========================================
# 2. LOAD MODEL AND GROUND TRUTH
# ==========================================

model = RTDETR(str(BEST_PT))
coco_gt = COCO(str(GT_JSON))

images = sorted(
    coco_gt.dataset["images"],
    key=lambda img: str(img["id"])
)

print(f"RT-DETR-L — FOLD {FOLD} OFFICIAL EVALUATION")
print("=" * 55)
print("Validation images:", len(images))
print("Image size: 960")
print("Confidence threshold: 0.01")
print("Max detections: 500")
print("GPU:", torch.cuda.get_device_name(0))

# ==========================================
# 3. GPU INFERENCE
# ==========================================

start = time.perf_counter()

all_detections = []
chunk_size = 50

for chunk_start in range(0, len(images), chunk_size):

    chunk_idx = chunk_start // chunk_size
    batch_images = images[
        chunk_start:chunk_start + chunk_size
    ]

    chunk_path = CHUNK_DIR / f"chunk_{chunk_idx:03d}.json"
    image_ids = [img["id"] for img in batch_images]

    if chunk_path.exists():
        with open(chunk_path) as f:
            saved = json.load(f)

        assert saved["image_ids"] == image_ids
        detections = saved["detections"]

    else:
        detections = []

        for img in batch_images:

            image_path = PREP_DIR / img["file_name"]
            assert image_path.exists()

            result = model.predict(
                source=str(image_path),
                imgsz=960,
                conf=0.01,
                iou=0.7,
                max_det=500,
                device=0,
                verbose=False
            )[0]

            height, width = result.orig_shape

            assert width == img["width"]
            assert height == img["height"]

            for box in result.boxes:

                cls = int(box.cls.item())
                score = float(box.conf.item())

                x1, y1, x2, y2 = box.xyxy[0].tolist()

                x1 = max(0.0, min(x1, width))
                x2 = max(0.0, min(x2, width))
                y1 = max(0.0, min(y1, height))
                y2 = max(0.0, min(y2, height))

                w = x2 - x1
                h = y2 - y1

                if w <= 0 or h <= 0:
                    continue

                detections.append({
                    "image_id": img["id"],
                    "category_id": cls + 1,
                    "bbox": [x1, y1, w, h],
                    "score": score
                })

        # Save chunk safely
        temp_path = chunk_path.with_suffix(".json.tmp")

        with open(temp_path, "w") as f:
            json.dump({
                "image_ids": image_ids,
                "detections": detections
            }, f)

        os.replace(temp_path, chunk_path)

    all_detections.extend(detections)

    print(
        f"Processed "
        f"{min(chunk_start + chunk_size, len(images))}"
        f"/{len(images)} images"
    )

with open(PRED_JSON, "w") as f:
    json.dump(all_detections, f)

inference_seconds = time.perf_counter() - start

print("\nINFERENCE COMPLETED")
print("Detections:", len(all_detections))
print("Inference time:", round(inference_seconds, 1), "s")

# ==========================================
# 4. COCO EVALUATION
# ==========================================

assert all_detections, "No predictions generated"

coco_dt = coco_gt.loadRes(str(PRED_JSON))

evaluator = COCOeval(coco_gt, coco_dt, "bbox")

evaluator.params.imgIds = [
    img["id"] for img in images
]

evaluator.params.maxDets = [1, 100, 500]

evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

# ==========================================
# 5. COMPUTE OFFICIAL METRICS
# ==========================================

precision = evaluator.eval["precision"]

def calculate_ap(area_idx=0, maxdet_idx=2, iou_idx=None):

    values = precision

    if iou_idx is not None:
        values = values[iou_idx:iou_idx + 1]

    values = values[:, :, :, area_idx, maxdet_idx]
    valid = values[values > -1]

    return float(valid.mean()) if valid.size else None

metrics = {
    "model": "RT-DETR-L",
    "experiment": EXPERIMENT,
    "mode": "cv",
    "fold": FOLD,
    "epochs": 4,
    "image_size": 960,
    "validation_images": len(images),
    "detections": len(all_detections),
    "mAP_maxDets100": calculate_ap(0, 1),
    "mAP_maxDets500": calculate_ap(0, 2),
    "AP50_maxDets500": calculate_ap(0, 2, 0),
    "AP_small": calculate_ap(1, 2),
    "AP_medium": calculate_ap(2, 2),
    "AP_large": calculate_ap(3, 2)
}

with open(METRICS_JSON, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nRT-DETR-L FOLD {FOLD} COCO RESULTS")
print("=" * 55)

for key, value in metrics.items():
    print(f"{key}: {value}")

print("\nMetrics saved:", METRICS_JSON)
print("COCO EVALUATION COMPLETED!")

loading annotations into memory...
Done (t=0.17s)
creating index...
index created!
RT-DETR-L — FOLD 1 OFFICIAL EVALUATION
Validation images: 992
Image size: 960
Confidence threshold: 0.01
Max detections: 500
GPU: Tesla T4
Processed 50/992 images
Processed 100/992 images
Processed 150/992 images
Processed 200/992 images
Processed 250/992 images
Processed 300/992 images
Processed 350/992 images
Processed 400/992 images
Processed 450/992 images
Processed 500/992 images
Processed 550/992 images
Processed 600/992 images
Processed 650/992 images
Processed 700/992 images
Processed 750/992 images
Processed 800/992 images
Processed 850/992 images
Processed 900/992 images
Processed 950/992 images
Processed 992/992 images

INFERENCE COMPLETED
Detections: 297600
Inference time: 119.1 s
Loading and preparing results...
DONE (t=2.74s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=89.64s).
Accumulating evaluation results...
DONE (t=5.11s).
 A

In [ ]:
import gc
import torch
import pandas as pd
from pathlib import Path
from ultralytics import RTDETR

# ==========================================
# 1. CONFIGURATION
# ==========================================

FOLD = 2
EXPERIMENT = "baseline"
TARGET_EPOCHS = 4

RESULTS_DIR = Path(
    "/content/drive_rtdetr_new/MyDrive/hw2_rtdetr"
)

CONFIG_DIR = RESULTS_DIR / "configs"

DATA_YAML = CONFIG_DIR / f"rtdetr_fold{FOLD}_cv.yaml"
WEIGHTS_PATH = RESULTS_DIR / "rtdetr-l.pt"

PROJECT_DIR = RESULTS_DIR / "cv_runs"
RUN_NAME = f"{EXPERIMENT}_cv_fold{FOLD}"
RUN_DIR = PROJECT_DIR / RUN_NAME

LAST_PT = RUN_DIR / "weights/last.pt"
BEST_PT = RUN_DIR / "weights/best.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

# ==========================================
# 2. VALIDATION
# ==========================================

assert torch.cuda.is_available(), "GPU is not available!"
assert DATA_YAML.exists(), f"Missing YAML: {DATA_YAML}"
assert WEIGHTS_PATH.exists(), "Pretrained weights not found!"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

completed_epochs = 0

if RESULTS_CSV.exists():
    completed_epochs = len(pd.read_csv(RESULTS_CSV))

print("RT-DETR-L BASELINE — FOLD 2")
print("=" * 55)

print("Fold:", FOLD)
print("Experiment:", EXPERIMENT)
print("Completed epochs:", completed_epochs)
print("Target epochs:", TARGET_EPOCHS)
print("GPU:", torch.cuda.get_device_name(0))
print("Image size: 960")
print("Batch size: 1")
print("Training YAML:", DATA_YAML)
print("Output directory:", RUN_DIR)

# ==========================================
# 3. TRAIN OR RESUME
# ==========================================

if completed_epochs >= TARGET_EPOCHS and BEST_PT.exists():

    print("\nFold 2 already completed. Skipping training.")

elif LAST_PT.exists():

    print("\nResuming Fold 2 from existing checkpoint...")

    model = RTDETR(str(LAST_PT))
    model.train(resume=True)

else:

    print("\nStarting Fold 2 from pretrained RT-DETR-L...")

    # Always start each fold from original pretrained weights
    model = RTDETR(str(WEIGHTS_PATH))

    model.train(
        data=str(DATA_YAML),
        imgsz=960,
        epochs=TARGET_EPOCHS,
        batch=1,
        device=0,
        optimizer="AdamW",
        lr0=0.001,
        seed=42,
        deterministic=False,
        amp=True,
        scale=0.5,
        workers=2,
        cache=False,
        val=False,
        save=True,
        save_period=1,
        plots=False,
        project=str(PROJECT_DIR),
        name=RUN_NAME,
        exist_ok=True
    )

# ==========================================
# 4. VERIFY TRAINING RESULTS
# ==========================================

print("\nRT-DETR-L FOLD 2 TRAINING STATUS")
print("=" * 55)

print("Results CSV:", RESULTS_CSV.exists())
print("Best checkpoint:", BEST_PT.exists())
print("Last checkpoint:", LAST_PT.exists())

if RESULTS_CSV.exists():
    results = pd.read_csv(RESULTS_CSV)

    print("Completed epochs:", len(results))

    display(results.tail(4))

    assert len(results) == TARGET_EPOCHS

assert BEST_PT.exists(), "Best checkpoint was not saved!"

print("\nFOLD 2 TRAINING COMPLETED SUCCESSFULLY!")

# ==========================================
# 5. CLEAN GPU MEMORY
# ==========================================

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

print("GPU cache cleared.")

RT-DETR-L BASELINE — FOLD 2
Fold: 2
Experiment: baseline
Completed epochs: 0
Target epochs: 4
GPU: Tesla T4
Image size: 960
Batch size: 1
Training YAML: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/configs/rtdetr_fold2_cv.yaml
Output directory: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/cv_runs/baseline_cv_fold2

Starting Fold 2 from pretrained RT-DETR-L...
New https://pypi.org/project/ultralytics/8.4.207 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive_rtdetr_new/MyDrive/hw2_rtdetr/configs/rtdetr_fold2_cv.yaml, degrees=0.0, deterministic=False, device=0, df

,epoch,time,train/giou_loss,train/cls_loss,train/l1_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/giou_loss,val/cls_loss,val/l1_loss,lr/pg0,lr/pg1,lr/pg2
0,1,531.657,1.60850,0.36898,0.33869,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.000333,0.000333,0.067016
1,2,1075.950,1.13143,0.60264,0.15410,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.000502,0.000502,0.033851
2,3,1632.370,0.88624,0.69612,0.12082,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.000505,0.000505,0.000521
3,4,2259.600,0.72054,0.76391,0.10054,0.25079,0.34844,0.11994,0.05824,0.75019,0.78749,0.11805,0.000258,0.000258,0.000258



FOLD 2 TRAINING COMPLETED SUCCESSFULLY!
GPU cache cleared.


In [ ]:
import json
import os
import time
import torch
import numpy as np

from pathlib import Path
from ultralytics import RTDETR
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ==========================================
# 1. CONFIGURATION
# ==========================================

FOLD = 2
EXPERIMENT = "baseline"

RESULTS_DIR = Path(
    "/content/drive_rtdetr_new/MyDrive/hw2_rtdetr"
)

PREP_DIR = Path("/content/hw2/data_prepared")

RUN_DIR = (
    RESULTS_DIR / "cv_runs" /
    f"{EXPERIMENT}_cv_fold{FOLD}"
)

BEST_PT = RUN_DIR / "weights/best.pt"
GT_JSON = PREP_DIR / f"coco/fold{FOLD}_val.json"

CHUNK_DIR = RUN_DIR / "prediction_chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

PRED_JSON = (
    RESULTS_DIR /
    f"{EXPERIMENT}_cv_fold{FOLD}_val_dets.json"
)

METRICS_JSON = (
    RESULTS_DIR /
    f"{EXPERIMENT}_cv_fold{FOLD}_metrics.json"
)

assert BEST_PT.exists()
assert GT_JSON.exists()
assert torch.cuda.is_available()

# ==========================================
# 2. LOAD MODEL AND GROUND TRUTH
# ==========================================

model = RTDETR(str(BEST_PT))
coco_gt = COCO(str(GT_JSON))

images = sorted(
    coco_gt.dataset["images"],
    key=lambda img: str(img["id"])
)

assert len(images) == 1285

print("RT-DETR-L — FOLD 2 COCO EVALUATION")
print("=" * 55)
print("Checkpoint:", BEST_PT)
print("Validation images:", len(images))
print("Image size:", 960)
print("Confidence:", 0.01)
print("Max detections:", 500)
print("GPU:", torch.cuda.get_device_name(0))

# ==========================================
# 3. INFERENCE WITH RESUMABLE CHUNKS
# ==========================================

start = time.perf_counter()

all_detections = []
chunk_size = 50

for chunk_start in range(0, len(images), chunk_size):

    chunk_idx = chunk_start // chunk_size

    chunk_images = images[
        chunk_start:chunk_start + chunk_size
    ]

    image_ids = [
        img["id"] for img in chunk_images
    ]

    chunk_path = (
        CHUNK_DIR / f"chunk_{chunk_idx:03d}.json"
    )

    if chunk_path.exists():

        with open(chunk_path) as f:
            saved = json.load(f)

        assert saved["image_ids"] == image_ids

        detections = saved["detections"]

    else:

        detections = []

        for img in chunk_images:

            image_path = PREP_DIR / img["file_name"]
            assert image_path.exists()

            result = model.predict(
                source=str(image_path),
                imgsz=960,
                conf=0.01,
                iou=0.7,
                max_det=500,
                device=0,
                verbose=False
            )[0]

            height, width = result.orig_shape

            assert width == img["width"]
            assert height == img["height"]

            for box in result.boxes:

                cls = int(box.cls.item())
                score = float(box.conf.item())

                assert 0 <= cls < 8

                x1, y1, x2, y2 = (
                    box.xyxy[0].tolist()
                )

                x1 = max(0.0, min(x1, width))
                x2 = max(0.0, min(x2, width))

                y1 = max(0.0, min(y1, height))
                y2 = max(0.0, min(y2, height))

                w = x2 - x1
                h = y2 - y1

                if w <= 0 or h <= 0:
                    continue

                detections.append({
                    "image_id": img["id"],
                    "category_id": cls + 1,
                    "bbox": [x1, y1, w, h],
                    "score": score
                })

        # Save each completed chunk
        temp_path = chunk_path.with_suffix(".json.tmp")

        with open(temp_path, "w") as f:
            json.dump({
                "image_ids": image_ids,
                "detections": detections
            }, f)

        os.replace(temp_path, chunk_path)

    all_detections.extend(detections)

    print(
        f"Processed "
        f"{min(chunk_start + chunk_size, len(images))}"
        f"/{len(images)} images"
    )

# Save combined predictions
with open(PRED_JSON, "w") as f:
    json.dump(all_detections, f)

inference_seconds = time.perf_counter() - start

print("\nINFERENCE COMPLETED")
print("=" * 55)
print("Images:", len(images))
print("Detections:", len(all_detections))
print("Inference time:", round(inference_seconds, 1), "s")

# ==========================================
# 4. COCO EVALUATION
# ==========================================

assert len(all_detections) > 0

coco_dt = coco_gt.loadRes(str(PRED_JSON))

evaluator = COCOeval(
    coco_gt,
    coco_dt,
    "bbox"
)

evaluator.params.imgIds = [
    img["id"] for img in images
]

evaluator.params.maxDets = [1, 100, 500]

evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

# ==========================================
# 5. EXTRACT COCO METRICS
# ==========================================

precision = evaluator.eval["precision"]

def calculate_ap(
    area_idx=0,
    maxdet_idx=2,
    iou_idx=None
):

    values = precision

    if iou_idx is not None:
        values = values[
            iou_idx:iou_idx + 1
        ]

    values = values[
        :, :, :, area_idx, maxdet_idx
    ]

    valid = values[values > -1]

    return (
        float(valid.mean())
        if valid.size
        else None
    )

metrics = {
    "model": "RT-DETR-L",
    "experiment": EXPERIMENT,
    "mode": "cv",
    "fold": FOLD,
    "epochs": 4,
    "image_size": 960,
    "validation_images": len(images),
    "detections": len(all_detections),
    "mAP_maxDets100": calculate_ap(0, 1),
    "mAP_maxDets500": calculate_ap(0, 2),
    "AP50_maxDets500": calculate_ap(0, 2, 0),
    "AP_small": calculate_ap(1, 2),
    "AP_medium": calculate_ap(2, 2),
    "AP_large": calculate_ap(3, 2)
}

# Save metrics
with open(METRICS_JSON, "w") as f:
    json.dump(metrics, f, indent=2)

# ==========================================
# 6. RESULTS
# ==========================================

print("\nRT-DETR-L FOLD 2 COCO RESULTS")
print("=" * 55)

for key, value in metrics.items():
    print(f"{key}: {value}")

print("\nPredictions saved:", PRED_JSON)
print("Metrics saved:", METRICS_JSON)

print("\nFOLD 2 COCO EVALUATION COMPLETED!")

loading annotations into memory...
Done (t=0.27s)
creating index...
index created!
RT-DETR-L — FOLD 2 COCO EVALUATION
Checkpoint: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/cv_runs/baseline_cv_fold2/weights/best.pt
Validation images: 1285
Image size: 960
Confidence: 0.01
Max detections: 500
GPU: Tesla T4
Processed 50/1285 images
Processed 100/1285 images
Processed 150/1285 images
Processed 200/1285 images
Processed 250/1285 images
Processed 300/1285 images
Processed 350/1285 images
Processed 400/1285 images
Processed 450/1285 images
Processed 500/1285 images
Processed 550/1285 images
Processed 600/1285 images
Processed 650/1285 images
Processed 700/1285 images
Processed 750/1285 images
Processed 800/1285 images
Processed 850/1285 images
Processed 900/1285 images
Processed 950/1285 images
Processed 1000/1285 images
Processed 1050/1285 images
Processed 1100/1285 images
Processed 1150/1285 images
Processed 1200/1285 images
Processed 1250/1285 images
Processed 1285/1285 images

INFERENCE

In [ ]:
import gc
import torch
import pandas as pd
import yaml

from pathlib import Path
from ultralytics import RTDETR

# ==========================================
# 1. CONFIGURATION
# ==========================================

FOLD = 3
EXPERIMENT = "baseline"
TARGET_EPOCHS = 4

RESULTS_DIR = Path(
    "/content/drive_rtdetr_new/MyDrive/hw2_rtdetr"
)

CONFIG_DIR = RESULTS_DIR / "configs"

DATA_YAML = CONFIG_DIR / f"rtdetr_fold{FOLD}_cv.yaml"
WEIGHTS_PATH = RESULTS_DIR / "rtdetr-l.pt"

PROJECT_DIR = RESULTS_DIR / "cv_runs"
RUN_NAME = f"{EXPERIMENT}_cv_fold{FOLD}"

RUN_DIR = PROJECT_DIR / RUN_NAME

LAST_PT = RUN_DIR / "weights/last.pt"
BEST_PT = RUN_DIR / "weights/best.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

# ==========================================
# 2. VALIDATION
# ==========================================

assert torch.cuda.is_available(), "GPU not available!"
assert DATA_YAML.exists(), f"Missing YAML: {DATA_YAML}"
assert WEIGHTS_PATH.exists(), "Pretrained weights missing!"

with open(DATA_YAML) as f:
    config = yaml.safe_load(f)

train_count = len(
    Path(config["train"]).read_text().splitlines()
)

val_count = len(
    Path(config["val"]).read_text().splitlines()
)

assert train_count == 2191
assert val_count == 1080

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

completed_epochs = 0

if RESULTS_CSV.exists():
    completed_epochs = len(pd.read_csv(RESULTS_CSV))

print("RT-DETR-L BASELINE — FOLD 3")
print("=" * 55)

print("Fold:", FOLD)
print("Experiment:", EXPERIMENT)
print("Training images:", train_count)
print("Validation images:", val_count)
print("Completed epochs:", completed_epochs)
print("Target epochs:", TARGET_EPOCHS)
print("GPU:", torch.cuda.get_device_name(0))
print("Image size: 960")
print("Batch size: 1")
print("Output directory:", RUN_DIR)

# ==========================================
# 3. TRAIN OR RESUME
# ==========================================

if completed_epochs >= TARGET_EPOCHS and BEST_PT.exists():

    print("\nFold 3 already completed. Skipping.")

elif LAST_PT.exists():

    print("\nResuming Fold 3 from checkpoint...")

    model = RTDETR(str(LAST_PT))
    model.train(resume=True)

elif completed_epochs > 0:

    raise RuntimeError(
        "Partial training history exists, but last.pt "
        "is missing. Check the saved files before restarting."
    )

else:

    print("\nStarting Fold 3 from pretrained RT-DETR-L...")

    # Independent initialization for each CV fold
    model = RTDETR(str(WEIGHTS_PATH))

    model.train(
        data=str(DATA_YAML),
        imgsz=960,
        epochs=TARGET_EPOCHS,
        batch=1,
        device=0,
        optimizer="AdamW",
        lr0=0.001,
        seed=42,
        deterministic=False,
        amp=True,
        scale=0.5,
        workers=2,
        cache=False,
        val=False,
        save=True,
        save_period=1,
        plots=False,
        project=str(PROJECT_DIR),
        name=RUN_NAME,
        exist_ok=True
    )

# ==========================================
# 4. VERIFY TRAINING RESULTS
# ==========================================

print("\nRT-DETR-L FOLD 3 TRAINING STATUS")
print("=" * 55)

print("Results CSV:", RESULTS_CSV.exists())
print("Best checkpoint:", BEST_PT.exists())
print("Last checkpoint:", LAST_PT.exists())

if RESULTS_CSV.exists():

    results = pd.read_csv(RESULTS_CSV)

    print("Completed epochs:", len(results))

    display(results.tail(4))

    assert len(results) == TARGET_EPOCHS

assert BEST_PT.exists(), "Best checkpoint missing!"

print("\nFOLD 3 TRAINING COMPLETED SUCCESSFULLY!")

# ==========================================
# 5. CLEAN GPU MEMORY
# ==========================================

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

print("GPU cache cleared.")

RT-DETR-L BASELINE — FOLD 3
Fold: 3
Experiment: baseline
Training images: 2191
Validation images: 1080
Completed epochs: 0
Target epochs: 4
GPU: Tesla T4
Image size: 960
Batch size: 1
Output directory: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/cv_runs/baseline_cv_fold3

Starting Fold 3 from pretrained RT-DETR-L...
New https://pypi.org/project/ultralytics/8.4.208 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive_rtdetr_new/MyDrive/hw2_rtdetr/configs/rtdetr_fold3_cv.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=No

,epoch,time,train/giou_loss,train/cls_loss,train/l1_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/giou_loss,val/cls_loss,val/l1_loss,lr/pg0,lr/pg1,lr/pg2
0,1,583.861,1.63768,0.37444,0.33780,0.00000,0.00000,0.0000,0.00000,0.00000,0.00000,0.00000,0.000333,0.000333,0.067015
1,2,1163.220,1.52173,0.38672,0.25371,0.00000,0.00000,0.0000,0.00000,0.00000,0.00000,0.00000,0.000502,0.000502,0.033850
2,3,1746.400,0.87016,0.72258,0.11914,0.00000,0.00000,0.0000,0.00000,0.00000,0.00000,0.00000,0.000505,0.000505,0.000520
3,4,2389.360,0.67778,0.79836,0.09355,0.13816,0.27294,0.0883,0.04037,0.95345,0.70012,0.17351,0.000258,0.000258,0.000258



FOLD 3 TRAINING COMPLETED SUCCESSFULLY!
GPU cache cleared.


In [ ]:
import json
import os
import time
import torch
import numpy as np

from pathlib import Path
from ultralytics import RTDETR
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ==========================================
# 1. CONFIGURATION
# ==========================================

FOLD = 3
EXPERIMENT = "baseline"

RESULTS_DIR = Path(
    "/content/drive_rtdetr_new/MyDrive/hw2_rtdetr"
)
PREP_DIR = Path("/content/hw2/data_prepared")

RUN_DIR = (
    RESULTS_DIR / "cv_runs" /
    f"{EXPERIMENT}_cv_fold{FOLD}"
)

BEST_PT = RUN_DIR / "weights/best.pt"
GT_JSON = PREP_DIR / f"coco/fold{FOLD}_val.json"

CHUNK_DIR = RUN_DIR / "prediction_chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

PRED_JSON = (
    RESULTS_DIR /
    f"{EXPERIMENT}_cv_fold{FOLD}_val_dets.json"
)

METRICS_JSON = (
    RESULTS_DIR /
    f"{EXPERIMENT}_cv_fold{FOLD}_metrics.json"
)

assert BEST_PT.is_file()
assert GT_JSON.is_file()
assert torch.cuda.is_available()

# ==========================================
# 2. LOAD MODEL AND GROUND TRUTH
# ==========================================

model = RTDETR(str(BEST_PT))
coco_gt = COCO(str(GT_JSON))

images = sorted(
    coco_gt.dataset["images"],
    key=lambda img: str(img["id"])
)

assert len(images) == 1080

print("RT-DETR-L — FOLD 3 COCO EVALUATION")
print("=" * 55)
print("Checkpoint:", BEST_PT)
print("Validation images:", len(images))
print("Image size: 960")
print("Confidence: 0.01")
print("Max detections: 500")
print("GPU:", torch.cuda.get_device_name(0))

# ==========================================
# 3. INFERENCE WITH RESUMABLE CHUNKS
# ==========================================

start = time.perf_counter()
all_detections = []
chunk_size = 50

for chunk_start in range(0, len(images), chunk_size):

    chunk_idx = chunk_start // chunk_size
    chunk_images = images[
        chunk_start:chunk_start + chunk_size
    ]

    image_ids = [img["id"] for img in chunk_images]
    chunk_path = CHUNK_DIR / f"chunk_{chunk_idx:03d}.json"

    if chunk_path.exists():

        with open(chunk_path) as f:
            saved = json.load(f)

        assert saved["image_ids"] == image_ids
        detections = saved["detections"]

    else:

        detections = []

        for img in chunk_images:

            image_path = PREP_DIR / img["file_name"]
            assert image_path.is_file(), image_path

            result = model.predict(
                source=str(image_path),
                imgsz=960,
                conf=0.01,
                iou=0.7,
                max_det=500,
                device=0,
                verbose=False
            )[0]

            height, width = result.orig_shape

            assert width == img["width"]
            assert height == img["height"]

            for box in result.boxes:

                cls = int(box.cls.item())
                score = float(box.conf.item())

                assert 0 <= cls < 8

                x1, y1, x2, y2 = box.xyxy[0].tolist()

                x1 = max(0.0, min(x1, width))
                x2 = max(0.0, min(x2, width))
                y1 = max(0.0, min(y1, height))
                y2 = max(0.0, min(y2, height))

                w = x2 - x1
                h = y2 - y1

                if w <= 0 or h <= 0:
                    continue

                detections.append({
                    "image_id": img["id"],
                    "category_id": cls + 1,
                    "bbox": [x1, y1, w, h],
                    "score": score
                })

        # Save finished chunk safely
        temp_path = chunk_path.with_suffix(".json.tmp")

        with open(temp_path, "w") as f:
            json.dump({
                "image_ids": image_ids,
                "detections": detections
            }, f)

        os.replace(temp_path, chunk_path)

    all_detections.extend(detections)

    print(
        f"Processed "
        f"{min(chunk_start + chunk_size, len(images))}"
        f"/{len(images)} images"
    )

# Save combined predictions
with open(PRED_JSON, "w") as f:
    json.dump(all_detections, f)

inference_seconds = time.perf_counter() - start

print("\nINFERENCE COMPLETED")
print("=" * 55)
print("Images:", len(images))
print("Detections:", len(all_detections))
print("Inference time:", round(inference_seconds, 1), "s")

# ==========================================
# 4. OFFICIAL COCO EVALUATION
# ==========================================

assert len(all_detections) > 0

coco_dt = coco_gt.loadRes(str(PRED_JSON))

evaluator = COCOeval(coco_gt, coco_dt, "bbox")
evaluator.params.imgIds = [img["id"] for img in images]
evaluator.params.maxDets = [1, 100, 500]

evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

# ==========================================
# 5. EXTRACT METRICS
# ==========================================

precision = evaluator.eval["precision"]

def calculate_ap(area_idx=0, maxdet_idx=2, iou_idx=None):

    values = precision

    if iou_idx is not None:
        values = values[iou_idx:iou_idx + 1]

    values = values[:, :, :, area_idx, maxdet_idx]
    valid = values[values > -1]

    return float(valid.mean()) if valid.size else None

metrics = {
    "model": "RT-DETR-L",
    "experiment": EXPERIMENT,
    "mode": "cv",
    "fold": FOLD,
    "epochs": 4,
    "image_size": 960,
    "validation_images": len(images),
    "detections": len(all_detections),
    "mAP_maxDets100": calculate_ap(0, 1),
    "mAP_maxDets500": calculate_ap(0, 2),
    "AP50_maxDets500": calculate_ap(0, 2, 0),
    "AP_small": calculate_ap(1, 2),
    "AP_medium": calculate_ap(2, 2),
    "AP_large": calculate_ap(3, 2)
}

with open(METRICS_JSON, "w") as f:
    json.dump(metrics, f, indent=2)

# ==========================================
# 6. RESULTS
# ==========================================

print("\nRT-DETR-L FOLD 3 COCO RESULTS")
print("=" * 55)

for key, value in metrics.items():
    print(f"{key}: {value}")

print("\nPredictions saved:", PRED_JSON)
print("Metrics saved:", METRICS_JSON)
print("\nFOLD 3 COCO EVALUATION COMPLETED!")

loading annotations into memory...
Done (t=0.20s)
creating index...
index created!
RT-DETR-L — FOLD 3 COCO EVALUATION
Checkpoint: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/cv_runs/baseline_cv_fold3/weights/best.pt
Validation images: 1080
Image size: 960
Confidence: 0.01
Max detections: 500
GPU: Tesla T4
Processed 50/1080 images
Processed 100/1080 images
Processed 150/1080 images
Processed 200/1080 images
Processed 250/1080 images
Processed 300/1080 images
Processed 350/1080 images
Processed 400/1080 images
Processed 450/1080 images
Processed 500/1080 images
Processed 550/1080 images
Processed 600/1080 images
Processed 650/1080 images
Processed 700/1080 images
Processed 750/1080 images
Processed 800/1080 images
Processed 850/1080 images
Processed 900/1080 images
Processed 950/1080 images
Processed 1000/1080 images
Processed 1050/1080 images
Processed 1080/1080 images

INFERENCE COMPLETED
Images: 1080
Detections: 324000
Inference time: 131.7 s
Loading and preparing results...
DONE (t=

In [ ]:
import gc
import torch
import pandas as pd
import yaml

from pathlib import Path
from ultralytics import RTDETR

# ==========================================
# 1. CONFIGURATION
# ==========================================

FOLD = 4
EXPERIMENT = "baseline"
TARGET_EPOCHS = 4

RESULTS_DIR = Path(
    "/content/drive_rtdetr_new/MyDrive/hw2_rtdetr"
)

CONFIG_DIR = RESULTS_DIR / "configs"

DATA_YAML = CONFIG_DIR / f"rtdetr_fold{FOLD}_cv.yaml"
WEIGHTS_PATH = RESULTS_DIR / "rtdetr-l.pt"

PROJECT_DIR = RESULTS_DIR / "cv_runs"
RUN_NAME = f"{EXPERIMENT}_cv_fold{FOLD}"
RUN_DIR = PROJECT_DIR / RUN_NAME

LAST_PT = RUN_DIR / "weights/last.pt"
BEST_PT = RUN_DIR / "weights/best.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

# ==========================================
# 2. VERIFY DATASET AND GPU
# ==========================================

assert torch.cuda.is_available(), "GPU unavailable!"
assert DATA_YAML.exists(), f"Missing YAML: {DATA_YAML}"
assert WEIGHTS_PATH.exists(), "Missing pretrained model!"

with open(DATA_YAML) as f:
    config = yaml.safe_load(f)

train_paths = [
    p for p in Path(config["train"]).read_text().splitlines()
    if p.strip()
]

val_paths = [
    p for p in Path(config["val"]).read_text().splitlines()
    if p.strip()
]

assert len(train_paths) == 2261
assert len(val_paths) == 940

assert all(Path(p).is_file() for p in train_paths)
assert all(Path(p).is_file() for p in val_paths)

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

completed_epochs = 0

if RESULTS_CSV.exists():
    completed_epochs = len(pd.read_csv(RESULTS_CSV))

print("RT-DETR-L BASELINE — FOLD 4")
print("=" * 55)

print("Training images:", len(train_paths))
print("Validation images:", len(val_paths))
print("Completed epochs:", completed_epochs)
print("Target epochs:", TARGET_EPOCHS)
print("Image size: 960")
print("Batch size: 1")
print("GPU:", torch.cuda.get_device_name(0))
print("Output:", RUN_DIR)

# ==========================================
# 3. TRAIN OR RESUME
# ==========================================

if completed_epochs >= TARGET_EPOCHS and BEST_PT.exists():

    print("\nFold 4 already completed. Skipping.")

elif LAST_PT.exists():

    print("\nResuming Fold 4 from checkpoint...")

    model = RTDETR(str(LAST_PT))
    model.train(resume=True)

elif completed_epochs > 0:

    raise RuntimeError(
        "Incomplete training results exist, but last.pt "
        "is missing. Check files before restarting."
    )

else:

    print("\nStarting Fold 4 from pretrained RT-DETR-L...")

    # Independent initialization:
    # Never use checkpoints from folds 0-3.
    model = RTDETR(str(WEIGHTS_PATH))

    model.train(
        data=str(DATA_YAML),
        imgsz=960,
        epochs=TARGET_EPOCHS,
        batch=1,
        device=0,
        optimizer="AdamW",
        lr0=0.001,
        seed=42,
        deterministic=False,
        amp=True,
        scale=0.5,
        workers=2,
        cache=False,
        val=False,
        save=True,
        save_period=1,
        plots=False,
        project=str(PROJECT_DIR),
        name=RUN_NAME,
        exist_ok=True
    )

# ==========================================
# 4. VERIFY CHECKPOINTS
# ==========================================

print("\nRT-DETR-L FOLD 4 TRAINING STATUS")
print("=" * 55)

print("Results CSV:", RESULTS_CSV.exists())
print("Best checkpoint:", BEST_PT.exists())
print("Last checkpoint:", LAST_PT.exists())

if RESULTS_CSV.exists():

    results = pd.read_csv(RESULTS_CSV)

    print("Completed epochs:", len(results))

    display(results.tail(4))

    assert len(results) == TARGET_EPOCHS

assert BEST_PT.exists(), "Best checkpoint not found!"

print("\nFOLD 4 TRAINING COMPLETED SUCCESSFULLY!")

# ==========================================
# 5. CLEAN GPU MEMORY
# ==========================================

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

print("GPU cache cleared.")

RT-DETR-L BASELINE — FOLD 4
Training images: 2261
Validation images: 940
Completed epochs: 0
Target epochs: 4
Image size: 960
Batch size: 1
GPU: Tesla T4
Output: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/cv_runs/baseline_cv_fold4

Starting Fold 4 from pretrained RT-DETR-L...
New https://pypi.org/project/ultralytics/8.4.208 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive_rtdetr_new/MyDrive/hw2_rtdetr/configs/rtdetr_fold4_cv.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropo

,epoch,time,train/giou_loss,train/cls_loss,train/l1_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/giou_loss,val/cls_loss,val/l1_loss,lr/pg0,lr/pg1,lr/pg2
0,1,566.51,2.14641,0.16915,0.56084,0.00000,0.00000,0.00000,0.00000,0.000,0.00000,0.00000,0.000333,0.000333,0.067015
1,2,1156.49,1.47021,0.44471,0.25572,0.00000,0.00000,0.00000,0.00000,0.000,0.00000,0.00000,0.000502,0.000502,0.033850
2,3,1760.33,1.13702,0.57816,0.15062,0.00000,0.00000,0.00000,0.00000,0.000,0.00000,0.00000,0.000505,0.000505,0.000520
3,4,2404.42,0.71580,0.79069,0.09748,0.15507,0.40636,0.14722,0.07422,0.698,0.77435,0.10483,0.000258,0.000258,0.000258



FOLD 4 TRAINING COMPLETED SUCCESSFULLY!
GPU cache cleared.


In [ ]:
import json
import os
import time
import torch
import numpy as np

from pathlib import Path
from ultralytics import RTDETR
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ==========================================
# 1. CONFIGURATION
# ==========================================

FOLD = 4
EXPERIMENT = "baseline"

RESULTS_DIR = Path(
    "/content/drive_rtdetr_new/MyDrive/hw2_rtdetr"
)
PREP_DIR = Path("/content/hw2/data_prepared")

RUN_DIR = (
    RESULTS_DIR / "cv_runs" /
    f"{EXPERIMENT}_cv_fold{FOLD}"
)

BEST_PT = RUN_DIR / "weights/best.pt"
GT_JSON = PREP_DIR / f"coco/fold{FOLD}_val.json"

CHUNK_DIR = RUN_DIR / "prediction_chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

PRED_JSON = (
    RESULTS_DIR /
    f"{EXPERIMENT}_cv_fold{FOLD}_val_dets.json"
)

METRICS_JSON = (
    RESULTS_DIR /
    f"{EXPERIMENT}_cv_fold{FOLD}_metrics.json"
)

assert BEST_PT.is_file(), "Best checkpoint missing!"
assert GT_JSON.is_file(), "COCO annotations missing!"
assert torch.cuda.is_available(), "GPU unavailable!"

# ==========================================
# 2. LOAD MODEL AND GROUND TRUTH
# ==========================================

model = RTDETR(str(BEST_PT))
coco_gt = COCO(str(GT_JSON))

images = sorted(
    coco_gt.dataset["images"],
    key=lambda img: str(img["id"])
)

assert len(images) == 940

print("RT-DETR-L — FOLD 4 COCO EVALUATION")
print("=" * 55)
print("Checkpoint:", BEST_PT)
print("Validation images:", len(images))
print("Image size: 960")
print("Confidence: 0.01")
print("Max detections: 500")
print("GPU:", torch.cuda.get_device_name(0))

# ==========================================
# 3. INFERENCE WITH RESUMABLE CHUNKS
# ==========================================

start = time.perf_counter()

all_detections = []
chunk_size = 50

for chunk_start in range(0, len(images), chunk_size):

    chunk_idx = chunk_start // chunk_size

    chunk_images = images[
        chunk_start:chunk_start + chunk_size
    ]

    image_ids = [img["id"] for img in chunk_images]

    chunk_path = (
        CHUNK_DIR / f"chunk_{chunk_idx:03d}.json"
    )

    if chunk_path.exists():

        with open(chunk_path) as f:
            saved = json.load(f)

        assert saved["image_ids"] == image_ids

        detections = saved["detections"]

    else:

        detections = []

        for img in chunk_images:

            image_path = PREP_DIR / img["file_name"]
            assert image_path.is_file(), image_path

            result = model.predict(
                source=str(image_path),
                imgsz=960,
                conf=0.01,
                iou=0.7,
                max_det=500,
                device=0,
                verbose=False
            )[0]

            height, width = result.orig_shape

            assert width == img["width"]
            assert height == img["height"]

            for box in result.boxes:

                cls = int(box.cls.item())
                score = float(box.conf.item())

                assert 0 <= cls < 8

                x1, y1, x2, y2 = (
                    box.xyxy[0].tolist()
                )

                x1 = max(0.0, min(x1, width))
                x2 = max(0.0, min(x2, width))

                y1 = max(0.0, min(y1, height))
                y2 = max(0.0, min(y2, height))

                w = x2 - x1
                h = y2 - y1

                if w <= 0 or h <= 0:
                    continue

                detections.append({
                    "image_id": img["id"],
                    "category_id": cls + 1,
                    "bbox": [x1, y1, w, h],
                    "score": score
                })

        # Save chunk safely
        temp_path = chunk_path.with_suffix(".json.tmp")

        with open(temp_path, "w") as f:
            json.dump({
                "image_ids": image_ids,
                "detections": detections
            }, f)

        os.replace(temp_path, chunk_path)

    all_detections.extend(detections)

    print(
        f"Processed "
        f"{min(chunk_start + chunk_size, len(images))}"
        f"/{len(images)} images"
    )

# ==========================================
# 4. SAVE COMBINED PREDICTIONS
# ==========================================

with open(PRED_JSON, "w") as f:
    json.dump(all_detections, f)

inference_seconds = time.perf_counter() - start

print("\nINFERENCE COMPLETED")
print("=" * 55)
print("Images:", len(images))
print("Detections:", len(all_detections))
print("Inference time:", round(inference_seconds, 1), "s")

# ==========================================
# 5. OFFICIAL COCO EVALUATION
# ==========================================

assert all_detections, "No predictions generated!"

coco_dt = coco_gt.loadRes(str(PRED_JSON))

evaluator = COCOeval(
    coco_gt,
    coco_dt,
    "bbox"
)

evaluator.params.imgIds = [
    img["id"] for img in images
]

evaluator.params.maxDets = [1, 100, 500]

evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

# ==========================================
# 6. EXTRACT METRICS
# ==========================================

precision = evaluator.eval["precision"]

def calculate_ap(
    area_idx=0,
    maxdet_idx=2,
    iou_idx=None
):

    values = precision

    if iou_idx is not None:
        values = values[
            iou_idx:iou_idx + 1
        ]

    values = values[
        :, :, :, area_idx, maxdet_idx
    ]

    valid = values[values > -1]

    return (
        float(valid.mean())
        if valid.size
        else None
    )

metrics = {
    "model": "RT-DETR-L",
    "experiment": EXPERIMENT,
    "mode": "cv",
    "fold": FOLD,
    "epochs": 4,
    "image_size": 960,
    "validation_images": len(images),
    "detections": len(all_detections),
    "mAP_maxDets100": calculate_ap(0, 1),
    "mAP_maxDets500": calculate_ap(0, 2),
    "AP50_maxDets500": calculate_ap(0, 2, 0),
    "AP_small": calculate_ap(1, 2),
    "AP_medium": calculate_ap(2, 2),
    "AP_large": calculate_ap(3, 2)
}

with open(METRICS_JSON, "w") as f:
    json.dump(metrics, f, indent=2)

# ==========================================
# 7. RESULTS
# ==========================================

print("\nRT-DETR-L FOLD 4 COCO RESULTS")
print("=" * 55)

for key, value in metrics.items():
    print(f"{key}: {value}")

print("\nPredictions saved:", PRED_JSON)
print("Metrics saved:", METRICS_JSON)

print("\nFOLD 4 COCO EVALUATION COMPLETED!")

loading annotations into memory...
Done (t=0.45s)
creating index...
index created!
RT-DETR-L — FOLD 4 COCO EVALUATION
Checkpoint: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/cv_runs/baseline_cv_fold4/weights/best.pt
Validation images: 940
Image size: 960
Confidence: 0.01
Max detections: 500
GPU: Tesla T4
Processed 50/940 images
Processed 100/940 images
Processed 150/940 images
Processed 200/940 images
Processed 250/940 images
Processed 300/940 images
Processed 350/940 images
Processed 400/940 images
Processed 450/940 images
Processed 500/940 images
Processed 550/940 images
Processed 600/940 images
Processed 650/940 images
Processed 700/940 images
Processed 750/940 images
Processed 800/940 images
Processed 850/940 images
Processed 900/940 images
Processed 940/940 images

INFERENCE COMPLETED
Images: 940
Detections: 282000
Inference time: 115.2 s
Loading and preparing results...
DONE (t=2.58s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox

In [ ]:
%pip install -q pycocotools

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive_rtdetr_new")

RESULTS_DIR = Path(
    "/content/drive_rtdetr_new/MyDrive/hw2_rtdetr"
)

print("RT-DETR-L CPU ENVIRONMENT CHECK")
print("=" * 55)

all_ready = True

for fold in range(5):

    gt_path = (
        RESULTS_DIR / "cv_annotations"
        / f"fold{fold}_val.json"
    )

    pred_path = (
        RESULTS_DIR
        / f"baseline_cv_fold{fold}_val_dets.json"
    )

    gt_ok = gt_path.is_file()
    pred_ok = pred_path.is_file()

    print(
        f"Fold {fold}: "
        f"Ground Truth = {gt_ok}, "
        f"Predictions = {pred_ok}"
    )

    all_ready = all_ready and gt_ok and pred_ok

assert all_ready, "Some files are missing!"

print("\nALL 5 FOLDS READY FOR CPU OOF EVALUATION!")

Mounted at /content/drive_rtdetr_new
RT-DETR-L CPU ENVIRONMENT CHECK
Fold 0: Ground Truth = True, Predictions = True
Fold 1: Ground Truth = True, Predictions = True
Fold 2: Ground Truth = True, Predictions = True
Fold 3: Ground Truth = True, Predictions = True
Fold 4: Ground Truth = True, Predictions = True

ALL 5 FOLDS READY FOR CPU OOF EVALUATION!


In [ ]:
import json
import gc
import shutil
import time
import numpy as np

from pathlib import Path
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ==========================================
# 1. PATHS
# ==========================================

RESULTS_DIR = Path(
    "/content/drive_rtdetr_new/MyDrive/hw2_rtdetr"
)
PREP_DIR = RESULTS_DIR / "cv_annotations"

LOCAL_DIR = Path("/content/rtdetr_oof")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

GT_LOCAL = LOCAL_DIR / "oof_gt.json"
PRED_LOCAL = LOCAL_DIR / "oof_dets.json"

GT_DRIVE = RESULTS_DIR / "baseline_oof_gt.json"
PRED_DRIVE = RESULTS_DIR / "baseline_oof_dets.json"
METRICS_PATH = RESULTS_DIR / "baseline_oof_metrics.json"

EXPECTED_VAL = [1142, 992, 1285, 1080, 940]

print("RT-DETR-L — 5-FOLD OOF EVALUATION")
print("=" * 55)

start = time.perf_counter()

# ==========================================
# 2. MERGE GROUND TRUTH AND PREDICTIONS
# ==========================================

merged_gt = {
    "info": {
        "description": "RT-DETR-L 5-fold VisDrone OOF"
    },
    "licenses": [],
    "images": [],
    "annotations": [],
    "categories": []
}

seen_names = set()
category_signature = None

next_image_id = 1
next_ann_id = 1
total_predictions = 0

with open(PRED_LOCAL, "w") as pred_out:
    pred_out.write("[")

    for fold in range(5):

        gt_path = PREP_DIR / f"fold{fold}_val.json"
        pred_path = (
            RESULTS_DIR /
            f"baseline_cv_fold{fold}_val_dets.json"
        )

        assert gt_path.is_file()
        assert pred_path.is_file()

        with open(gt_path) as f:
            gt = json.load(f)

        with open(pred_path) as f:
            predictions = json.load(f)

        assert len(gt["images"]) == EXPECTED_VAL[fold]
        assert len(predictions) == 300 * EXPECTED_VAL[fold]

        # Ensure category mapping is identical
        signature = sorted(
            (c["id"], c["name"])
            for c in gt["categories"]
        )

        if category_signature is None:
            category_signature = signature
            merged_gt["categories"] = gt["categories"]
        else:
            assert signature == category_signature

        # Unique global image IDs
        id_map = {}

        for img in gt["images"]:

            old_id = img["id"]
            name = Path(img["file_name"]).name

            assert old_id not in id_map
            assert name not in seen_names

            seen_names.add(name)
            id_map[old_id] = next_image_id

            img["id"] = next_image_id
            merged_gt["images"].append(img)

            next_image_id += 1

        # Unique global annotation IDs
        for ann in gt["annotations"]:

            assert ann["image_id"] in id_map

            ann["image_id"] = id_map[ann["image_id"]]
            ann["id"] = next_ann_id

            merged_gt["annotations"].append(ann)
            next_ann_id += 1

        # Remap prediction image IDs
        for det in predictions:
            assert det["image_id"] in id_map
            det["image_id"] = id_map[det["image_id"]]

        # Write this fold without holding all folds'
        # predictions in RAM simultaneously
        if fold > 0:
            pred_out.write(",")

        pred_out.write(
            json.dumps(
                predictions,
                separators=(",", ":")
            )[1:-1]
        )

        total_predictions += len(predictions)

        print(
            f"Fold {fold}: "
            f"{len(gt['images'])} images, "
            f"{len(predictions):,} predictions"
        )

        del gt, predictions
        gc.collect()

    pred_out.write("]")

# ==========================================
# 3. VALIDATE AND SAVE MERGED DATA
# ==========================================

assert len(merged_gt["images"]) == 5439
assert len(seen_names) == 5439
assert total_predictions == 1631700
assert len(merged_gt["categories"]) == 8

ignored_count = sum(
    ann.get("iscrowd", 0) == 1
    for ann in merged_gt["annotations"]
)

with open(GT_LOCAL, "w") as f:
    json.dump(merged_gt, f)

print("\nMERGED OOF DATASET")
print("=" * 55)
print("Images:", len(merged_gt["images"]))
print("Annotations:", len(merged_gt["annotations"]))
print("Ignored annotations:", ignored_count)
print("Predictions:", total_predictions)
print("Categories:", len(merged_gt["categories"]))

del merged_gt
gc.collect()

# Save merged data to Google Drive
shutil.copy2(GT_LOCAL, GT_DRIVE)
shutil.copy2(PRED_LOCAL, PRED_DRIVE)

print("\nMerged files saved to Google Drive.")

# ==========================================
# 4. GLOBAL COCO EVALUATION
# ==========================================

print("\nRUNNING GLOBAL OOF COCOeval")
print("=" * 55)

coco_gt = COCO(str(GT_LOCAL))
coco_dt = coco_gt.loadRes(str(PRED_LOCAL))

evaluator = COCOeval(
    coco_gt,
    coco_dt,
    "bbox"
)

evaluator.params.imgIds = sorted(
    coco_gt.getImgIds()
)

evaluator.params.maxDets = [1, 100, 500]

evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

# ==========================================
# 5. EXTRACT METRICS
# ==========================================

precision = evaluator.eval["precision"]

def calculate_ap(
    area_idx=0,
    maxdet_idx=2,
    iou_idx=None
):

    values = precision

    if iou_idx is not None:
        values = values[
            iou_idx:iou_idx + 1
        ]

    values = values[
        :, :, :, area_idx, maxdet_idx
    ]

    valid = values[values > -1]

    return (
        float(valid.mean())
        if valid.size
        else None
    )

metrics = {
    "model": "RT-DETR-L",
    "experiment": "baseline",
    "mode": "OOF",
    "folds": 5,
    "epochs_per_fold": 4,
    "image_size": 960,
    "images": 5439,
    "predictions": total_predictions,
    "mAP_maxDets100": calculate_ap(0, 1),
    "mAP_maxDets500": calculate_ap(0, 2),
    "AP50_maxDets500": calculate_ap(0, 2, 0),
    "AP75_maxDets500": calculate_ap(0, 2, 5),
    "AP_small": calculate_ap(1, 2),
    "AP_medium": calculate_ap(2, 2),
    "AP_large": calculate_ap(3, 2)
}

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

elapsed = time.perf_counter() - start

# ==========================================
# 6. FINAL RESULTS
# ==========================================

print("\nRT-DETR-L BASELINE — FINAL OOF RESULTS")
print("=" * 55)

for key, value in metrics.items():
    print(f"{key}: {value}")

print("\nTotal elapsed:", round(elapsed / 60, 2), "minutes")
print("Metrics saved:", METRICS_PATH)
print("\n5-FOLD OOF EVALUATION COMPLETED!")

RT-DETR-L — 5-FOLD OOF EVALUATION
Fold 0: 1142 images, 342,600 predictions
Fold 1: 992 images, 297,600 predictions
Fold 2: 1285 images, 385,500 predictions
Fold 3: 1080 images, 324,000 predictions
Fold 4: 940 images, 282,000 predictions

MERGED OOF DATASET
Images: 5439
Annotations: 355409
Ignored annotations: 109624
Predictions: 1631700
Categories: 8

Merged files saved to Google Drive.

RUNNING GLOBAL OOF COCOeval
loading annotations into memory...
Done (t=2.19s)
creating index...
index created!
Loading and preparing results...
DONE (t=23.38s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=908.05s).
Accumulating evaluation results...
DONE (t=54.14s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.051
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=500 ] = 0.105
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=500 ] = 0.046
 Average Precision  (AP) @[ IoU=0.50:0.95 

In [ ]:
import json
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(
    "/content/drive_rtdetr_new/MyDrive/hw2_rtdetr"
)

# Load official RT-DETR OOF metrics
with open(RESULTS_DIR / "baseline_oof_metrics.json") as f:
    rt = json.load(f)

# Previously obtained YOLO11n results
yolo = {
    "mAP_maxDets100": 0.148355,
    "AP50_maxDets500": 0.294611,
    "AP_small": 0.044544,
    "AP_medium": 0.145435,
    "AP_large": 0.318932
}

metric_keys = [
    "mAP_maxDets100",
    "AP50_maxDets500",
    "AP_small",
    "AP_medium",
    "AP_large"
]

comparison = pd.DataFrame({
    "Metric": metric_keys,
    "YOLO11n_best_CV": [yolo[k] for k in metric_keys],
    "RTDETR_L_baseline": [rt[k] for k in metric_keys]
})

comparison["Difference_YOLO_minus_RTDETR"] = (
    comparison["YOLO11n_best_CV"]
    - comparison["RTDETR_L_baseline"]
)

# Load all five RT-DETR folds
fold_results = []

for fold in range(5):
    path = RESULTS_DIR / f"baseline_cv_fold{fold}_metrics.json"

    with open(path) as f:
        metrics = json.load(f)

    fold_results.append(metrics)

fold_df = pd.DataFrame(fold_results)

# Save both tables
comparison_path = RESULTS_DIR / "model_cv_comparison.csv"
fold_path = RESULTS_DIR / "rtdetr_baseline_5fold_summary.csv"

comparison.to_csv(comparison_path, index=False)
fold_df.to_csv(fold_path, index=False)

print("FINAL MODEL COMPARISON")
print("=" * 55)
display(comparison.round(5))

print("\nRT-DETR-L FOLD RESULTS")
print("=" * 55)

display(
    fold_df[
        ["fold", "validation_images",
         "mAP_maxDets100", "AP50_maxDets500"]
    ].round(5)
)

assert fold_df["validation_images"].sum() == 5439

print("\nRESULTS SAVED SUCCESSFULLY!")
print("Comparison:", comparison_path)
print("Fold summary:", fold_path)

FINAL MODEL COMPARISON


,Metric,YOLO11n_best_CV,RTDETR_L_baseline,Difference_YOLO_minus_RTDETR
0,mAP_maxDets100,0.14835,0.05083,0.09753
1,AP50_maxDets500,0.29461,0.10457,0.19004
2,AP_small,0.04454,0.02081,0.02373
3,AP_medium,0.14544,0.05001,0.09542
4,AP_large,0.31893,0.10251,0.21642



RT-DETR-L FOLD RESULTS


,fold,validation_images,mAP_maxDets100,AP50_maxDets500
0,0,1142,0.06540,0.13949
1,1,992,0.05236,0.09749
2,2,1285,0.05789,0.12288
3,3,1080,0.04185,0.09147
4,4,940,0.07533,0.15081



RESULTS SAVED SUCCESSFULLY!
Comparison: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/model_cv_comparison.csv
Fold summary: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/rtdetr_baseline_5fold_summary.csv


In [ ]:
import pandas as pd
import yaml
from pathlib import Path

RESULTS_DIR = Path(
    "/content/drive_rtdetr_new/MyDrive/hw2_rtdetr"
)

rows = []

for fold in range(5):

    run_dir = (
        RESULTS_DIR / "cv_runs" /
        f"baseline_cv_fold{fold}"
    )

    csv_path = run_dir / "results.csv"
    args_path = run_dir / "args.yaml"

    assert csv_path.exists()
    assert args_path.exists()

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    with open(args_path) as f:
        args = yaml.safe_load(f)

    assert len(df) == 4

    for _, row in df.iterrows():

        rows.append({
            "fold": fold,
            "epoch": int(row["epoch"]),
            "giou_loss": row["train/giou_loss"],
            "cls_loss": row["train/cls_loss"],
            "l1_loss": row["train/l1_loss"],
            "lr_pg0": row["lr/pg0"],
            "lr_pg1": row["lr/pg1"],
            "lr_pg2": row["lr/pg2"],
            "configured_lr0": args["lr0"],
            "warmup_epochs": args["warmup_epochs"]
        })

diagnostics = pd.DataFrame(rows)

summary = (
    diagnostics.groupby("epoch")[
        ["giou_loss", "cls_loss", "l1_loss",
         "lr_pg0", "lr_pg1", "lr_pg2"]
    ]
    .mean()
    .reset_index()
)

print("RT-DETR-L TRAINING DIAGNOSTICS")
print("=" * 55)

print("Total folds:", diagnostics["fold"].nunique())
print("Training records:", len(diagnostics))

print("\nMEAN LOSSES AND LEARNING RATES BY EPOCH")
display(summary.round(6))

print("\nCONFIGURED TRAINING PARAMETERS")
print("Learning rate:", diagnostics["configured_lr0"].unique())
print("Warmup epochs:", diagnostics["warmup_epochs"].unique())

output_path = (
    RESULTS_DIR / "rtdetr_training_diagnostics.csv"
)

diagnostics.to_csv(output_path, index=False)

print("\nTRAINING DIAGNOSTICS SAVED!")
print("Path:", output_path)

RT-DETR-L TRAINING DIAGNOSTICS
Total folds: 5
Training records: 20

MEAN LOSSES AND LEARNING RATES BY EPOCH


,epoch,giou_loss,cls_loss,l1_loss,lr_pg0,lr_pg1,lr_pg2
0,1,1.722144,0.337902,0.386284,0.000333,0.000333,0.067015
1,2,1.325316,0.508912,0.207450,0.000502,0.000502,0.033850
2,3,0.951292,0.680384,0.129884,0.000505,0.000505,0.000520
3,4,0.721688,0.780116,0.099046,0.000258,0.000258,0.000258



CONFIGURED TRAINING PARAMETERS
Learning rate: [0.001]
Warmup epochs: [3.]

TRAINING DIAGNOSTICS SAVED!
Path: /content/drive_rtdetr_new/MyDrive/hw2_rtdetr/rtdetr_training_diagnostics.csv
